# 04 - Snapshot Validation and COM Smoke Runner

Validate `pptx_get_slide_snapshot` and run the full Windows COM smoke script from a notebook.

In [ ]:
import base64
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python").exists() and (REPO_ROOT.parent / "python").exists():
    REPO_ROOT = REPO_ROOT.parent
PYTHON_ROOT = REPO_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))


def build_helpers():
    from errors import BridgeError
    from service import PowerPointService

    return BridgeError, PowerPointService


BridgeError, PowerPointService = build_helpers()
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook-tests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
service = PowerPointService()
engine_info = service.dispatch("pptx_get_engine_info", {})
print("Engine:", engine_info.get("engine"))

create_result = service.dispatch("pptx_create_presentation", {"width": "10in", "height": "5.625in"})
presentation_id = create_result["presentation_id"]

layouts = service.dispatch("pptx_get_layouts", {"presentation_id": presentation_id})["layouts"]
layout_name = layouts[0]["name"]
add = service.dispatch("pptx_add_slide", {"presentation_id": presentation_id, "layout_name": layout_name})
slide_index = int(add["added_slide_index"])

# Best-effort set placeholder text for a richer snapshot.
placeholders = service.dispatch(
    "pptx_get_placeholders", {"presentation_id": presentation_id, "slide_index": slide_index}
)["placeholders"]
for ph in placeholders:
    try:
        service.dispatch(
            "pptx_set_placeholder_text",
            {
                "presentation_id": presentation_id,
                "slide_index": slide_index,
                "placeholder_name": ph["name"],
                "text_content": "Snapshot test text",
            },
        )
        break
    except BridgeError:
        continue

snapshot_path = (ARTIFACT_DIR / "nb04_snapshot.jpg").resolve()
try:
    snap = service.dispatch(
        "pptx_get_slide_snapshot",
        {
            "presentation_id": presentation_id,
            "slide_index": slide_index,
            "width_px": 1280,
        },
    )
    snapshot_path.write_bytes(base64.b64decode(snap["snapshot_base64"]))
    print("Snapshot saved:", snapshot_path)
except BridgeError as exc:
    if exc.code == "dependency_missing":
        print("Snapshot dependency missing:", exc.message)
        print(exc.details)
    else:
        raise

output_path = (ARTIFACT_DIR / "nb04_snapshot_output.pptx").resolve()
service.dispatch("pptx_save_presentation", {"presentation_id": presentation_id, "output_path": str(output_path)})
service.dispatch("pptx_close_presentation", {"presentation_id": presentation_id})
service.shutdown()
print("Presentation saved:", output_path)

Run the full Windows COM smoke script from notebook (recommended on Windows host):

In [ ]:
cmd = [
    sys.executable,
    str((REPO_ROOT / "scripts" / "windows_com_smoke.py").resolve()),
    "--output-dir",
    str((REPO_ROOT / "artifacts" / "com-smoke").resolve()),
]

result = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True)
print("Exit code:", result.returncode)
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    print("Smoke script failed. On non-Windows or without PowerPoint, this is expected.")

In [ ]:
report_dir = (REPO_ROOT / "artifacts" / "com-smoke").resolve()
json_reports = sorted(report_dir.glob("com_smoke_report_*.json"))
if json_reports:
    latest = json_reports[-1]
    print("Latest report:", latest)
    data = json.loads(latest.read_text(encoding="utf-8"))
    print("Report status:", data.get("status"))
    print("Engine:", data.get("engine"))
    print("Steps:", len(data.get("steps", [])))
else:
    print("No COM smoke reports found yet.")